In [ ]:
import plotting  # this has to be run from /scripts/cococo

import mqt.qecc.cococo.utils_routing as utils
from mqt.qecc.cococo import circuit_construction, layouts

## Example for Router with Movable Qubits

### Choose Layout

In [ ]:
layout_type = "triple"
m = 4
n = 4
factories = []
remove_edges = False
g, data_qubit_locs, factory_ring = layouts.gen_layout_scalable(layout_type, m, n, factories, remove_edges)
layout = dict(enumerate(data_qubit_locs))
t = 4  # mock because we have only cnots here in the example, but used later

In [ ]:
plotting.plot_lattice_paths(g, {}, {}, layout, factories, size=(18, 8))

### Choose Circuit

In [ ]:
q = len(data_qubit_locs)
j = 8
num_gates = q * 2
dag, pairs = circuit_construction.create_random_sequential_circuit_dag(
    j,
    q,
    num_gates,
)

### Run Standard Router with Standard Layout

In [ ]:
terminal_pairs = layouts.translate_layout_circuit(pairs, layout)  # let's stick to the simple layout

In [ ]:
router = utils.BasicRouter(g, data_qubit_locs, factories, valid_path="cc", t=t, metric="exact", use_dag=True)
layers = router.split_layer_terminal_pairs(terminal_pairs)
vdp_layers, _ = router.find_total_vdp_layers_dyn(layers, data_qubit_locs, router.factory_times, layout, testing=True)
print("Len of schedule without teleportation: ", len(vdp_layers))

### Run Movable Qubits Router with Standard Initial Layout

In [ ]:
router = utils.TeleportationRouter(
    g, data_qubit_locs, factories, valid_path="cc", t=t, metric="exact", use_dag=True, seed=1
)

max_iters = 100
T_start = 100.0
T_end = 0.1
alpha = 0.95
radius = 10
k_lookahead = 5
metric = "exact"

steiner_init_type = "full_random"
jump_harvesting = True
stimtest = True

reduce_steiner = True
idle_move_type = "later"

schedule, _ = router.optimize_layers(
    terminal_pairs,
    layout,
    max_iters,
    T_start,
    T_end,
    alpha,
    radius=radius,
    k_lookahead=k_lookahead,
    steiner_init_type=steiner_init_type,
    jump_harvesting=jump_harvesting,
    reduce_steiner=reduce_steiner,
    idle_move_type=idle_move_type,
    reduce_init_steiner=False,
    stimtest=stimtest,
)

In [ ]:
print("Len of schedule with teleport router: ", len(schedule))

In [ ]:
print("Reduction Delta: ", len(vdp_layers) - len(schedule))

In [ ]:
plotting.plot_schedule(g, schedule[:3], factories, size=(12, 4))